# Mixture of Experts

**Mixture of Experts**，即**混合专家模型**，在LLM领域(甚至它之外的领域，比如CV)可能会经常听到大家提起这个概念，如果要用一句话总结的话，其实就是在一个很大的前馈网络上进行分组，每一组叫做一个专家，把序列中的token通过路由到不同的专家进行处理，下面就来**简单**学习一下

<div align="center">
    <img src="./imgs/MoE_architecture.png" alt="MoE_architecture" style="width: 500px; height: auto;">
</div>

[参考知乎解析](https://zhuanlan.zhihu.com/p/81886457827)

[参考博客(图源)，也是内容的原始来源](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-mixture-of-experts)

[MAE Pytorch 参考实现](https://zhuanlan.zhihu.com/p/701777558)

## 理论部分

### 什么是MoE

简单来说，区别于**稠密模型(dense model，也就是不带MoE的传统模型)**，**MoE**就是把**Transformer**中的前馈网络分成了不同的部分，每一个部分叫做一个**专家(expert)**，并且在处理某一个序列时，通过一个**路由器(router)或者门控网络(gating network)**，把token分发给不同的专家(其实就是一个MLP)处理，它的基本结构如下图:

<div align="center">
    <img src="./imgs/MoE_multilayer.png" alt="MoE_multilayer" style="width: 500px; height: auto;">
</div>

我们虽然把每一个子模块叫做一个专家，但其实它并不是我们传统意义上的那种具有特殊领域知识的专家，它们的专长主要体现在**每个专家能够专注于处理一个序列(比如句子)中不同的语法部分**，比如专家1处理停顿符号(如逗号)，专家2处理动词(比如打)，专家3处理语气词(比如咕咕嘎嘎)，等等，如下图:

<div align="center">
    <img src="./imgs/MoE_layerprocess.png" alt="MoE_layerprocess" style="width: 500px; height: auto;">
</div>

而路由器/门控网络的作用，就是**为每一个token选择专家来对他进行处理**，本质就是在不同层选择不同的前馈网络(就是专家)对token进行处理，如下图，对于一个输入的token，路由机制为它在每一层分别选择了一个专家进行处理:
<div align="center">
    <img src="./imgs/MoE_router.png" alt="MoE_router" style="width: 500px; height: auto;">
</div>


### 专家模块

### 稠密模型

前面我们已经说过，专家本质上就是一个**前馈神经网路(或者看成一个MLP)**，并且在传统的模型中，这个网络的所有参数都会被激活，所以称之为**稠密模型(dense model)**，在一个**decoder-only Transformer**中，在对序列进行**因果掩码自注意力**之后，我们就需要使用一个前馈神经网络对处理的自注意力输出结果进行非线性映射，如下图所示:

<div align="center">
    <img src="./imgs/decoder-only_Transformer.png" alt="decoder-only_Transformer" style="width: 500px; height: auto;">
</div>

前馈神经网络的作用在于，把注意力模块捕捉到的上下文关系进行非线性的变换，从而捕捉数据间更加复杂的关系，而为了捕捉更为复杂的关系，前馈神经网络的中间层会对原始数据的维度进行拓展，从而能够捕捉更高维空间中的关系，最后再映射为原始维度，如下图所示:

<div align="center">
    <img src="./imgs/FFN.png" alt="FFN" style="width: 500px; height: auto;">
</div>


#### 稀疏模型

与稠密模型不同的是，**稀疏模型(sparse model)**在对输入进行作用时，只会激活部分的参数，就如我们之前所述，把一个大的前馈网络**分割成不同的组(就是专家)**，再把输入序列的不同token路由到不同的专家进行处理，这样在训练时，每个专家都能学到自己的“专长”，从而在推理时，为不同的token分配不同的专家处理，就是一种”**集各家所长**“的思想

之前我们也说过，每个专家的本质就是一个前馈网络，它们的专长体现在能够擅长对于不同特定类型的token进行处理，也就是**更关注语法结构而非特定专业领域的知识**，如下图所示:

<div align="center">
    <img src="./imgs/expert_pro.png" alt="expert_pro" style="width: 500px; height: auto;">
</div>

也就是说，相同类型的token会更倾向于路由到相同的专家中，比如标点都路由到第二层或第六层的某一个专家中进行处理，所以专家学习处理的是**某种特定类型(比如标点)的token而非专业领域的专长知识**


#### Expert的架构

我们之前说过，一个专家实质上就是一个前馈网络，MoE也就是把多个前馈网络组合在一起，从而能够分工对不同类型的token进行专门化处理，那么它的架构就显而易见了，就是替换原来的前馈网络为多个前馈网络(专家)的组合，架构如下图所示:

<div align="center">
    <img src="./imgs/expert_architecture.png" alt="expert_architecture" style="width: 700px; height: auto;">
</div>


### 路由模块

#### 路由器(Router)/门控网络(Gating Network)

其实，**路由器(也可以说是门控网络)**，本质上也是一个前馈网络，它的作用是为每一个token输出一个概率分布，每个概率表示选择某一个专家的概率，然后再根据概率的高低选择最匹配的专家，它的结构如下图所示:

<div align="center">
    <img src="./imgs/router_architecture.png" alt="router_architecture" style="width: 500px; height: auto;">
</div>

**注意**，图中有一个乘法运算，意思是说，把选择的专家的输出结果与这个token选择专家的概率进行加权平均，因为可能一个token会选择好几个专家对其进行处理，那么，我们就得到了最终的**MoE layer**的架构图，也就是把这个router+experts group的模块替换原来的前馈网络模块，如下图:

<div align="center">
    <img src="./imgs/final_decoder.png" alt="final_decoder" style="width: 500px; height: auto;">
</div>


#### 稀疏MoE和稠密MoE

简单来说就是，**稀疏MoE**对于每个token只选择少数几个专家进行计算，而**稠密MoE**对于每个token要激活所有的专家对其进行计算，而大多数LLM实现的MoE几乎都是**稀疏MoE**，稀疏MoE和稠密MoE的对比图如下:

<div align="center">
    <img src="./imgs/sparse_and_dense.png" alt="sparse_and_dense" style="width: 500px; height: auto;">
</div>

为什么稀疏MoE成为了更多LLM的选择呢？因为，所有专家对于某一个token的处理结果都需要使用这个token选择这些专家的概率进行平均，而大部分的专家可能被选择的概率很低，那么它们最后就会被权重给稀释掉，几乎不会对输出产生很大的影响，那么就可以只选择少数几个高概率专家对这个token进行处理，从而**降低推理成本**(因为大多数专家并没有参与对这个token的处理)


#### 专家的选择过程

如上文所述，对于某一个token，使用一个前馈网络对其进行处理，输出这个token对于每个专家的选择概率，使用**softmax**处理进行归一化操作(结果记为G(x))，然后选择几个专家对这个token进行处理(处理结果记为E(x))，最后使用选择这些专家的概率进行**加权平均汇总**，得到这个token的MoE layer的输出结果，如下图所示:

<div align="center">
    <img src="./imgs/MoE_process.png" alt="MoE_process" style="width: 500px; height: auto;">
</div>


### 负载均衡

我们已经弄清了MoE中的专家和路由两个部分，在实际训练时会出现一个问题，某些专家可能学习得更快，导致路由器总是倾向于选择它们，某些专家几乎得不到训练，使得最终的训练和推理出现偏倚和性能下降的问题，解决的方式是进行**负载均衡(load balancing)**，尽可能把所有token均分给每个专家，从而让每个专家都发挥应有的作用

#### KeepTopK策略

**KeepTopK**就是说，每个token只选择概率最高的K个专家对其进行处理，而防止某些专家总是被选择的一种方式就是在概率选择中引入一个可训练的高斯噪声(实现方式可以看代码部分)，避免模型总是选择相同的专家处理输入，如下图所示:

<div align="center">
    <img src="./imgs/add_noise1.png" alt="add_noise1" style="width: 500px; height: auto;">
</div>

之后同样**softmax**操作得到概率分布，再选择TopK概率的专家对这个token处理，再把每个专家的结果使用选择他们的概率进行加权平均得到这个token最终的处理结果

我们上述讨论的专家选择策略称为**Token Choice**，也就是让token给每个专家打分(分数就是选择某个专家的概率)，然后再用这些专家对这个token进行处理，再使用选择它们的概率加权平均得到最终这个token的处理结果

#### 辅助损失

为了实现负载均衡，我们引入了辅助损失，这个损失用来衡量每个专家的选择是否平均，也就是负载是否均衡，接下来就介绍两种负载均衡的辅助损失

##### 变异系数

这种损失的做法是:

首先把每个token输出的选择每个专家的概率分布相加，再计算这个分布的均值$\mu$和标准差$\sigma$，然后通过如下公式计算出变异系数:

$$
\text{Coefficient Variation(CV)} = \frac{\sigma}{\mu}
$$

流程如下图:

<div align="center">
    <img src="./imgs/CV.png" alt="CV" style="width: 500px; height: auto;">
</div>

如果CV很高，说明专家的选择概率并不均衡；反之，如果所有专家被选择的概率都差不多，那么CV就很低，也就达到了一种负载均衡的状态，如下图:

<div align="center">
    <img src="./imgs/CV_compare.png" alt="CV_compare" style="width: 500px; height: auto;">
</div>

这个损失很好理解，标准差$\sigma$就是用来衡量数据的变化幅度的，变化幅度大，标准差就大，也就意味着每个专家被选择的概率变化幅度大，即负载不均；反之，如果标准差小，每个专家被选择的概率变化幅度小，负载就均衡了，而除以的均值$\mu$可以理解为一个归一化因子，因为序列的token数和每个token选择的专家数可能会不同，导致均值不同，除以均值就是消除这个影响，从而公平比较它们的分布变化幅度


##### Switch Transformer中的辅助损失

在**Switch Transformer**中，作者的负载均衡辅助函数如下:

$$
\text{Auxiliary Loss} = \alpha \cdot N \sum_{i=1}^N P_i \cdot f_i
$$

其中:

- $\alpha$: 超参数，用于调整这个辅助函数在总损失函数中的权重，值过高会影响主要损失的训练，值太低则起不到负载均衡的作用

- $N$: 总专家数

- $P_i$: 第i个专家被选择的概率的平均值，即所有token选择第i个专家的概率的和的平均

- $f_i$: 第i个专家处理的token数占所有token数的比例


上面的损失在负载均衡(也就是每个专家被选择的概率都相同)时损失最小，所以优化时会让模型尽可能均匀地为每个专家分配token从而达到负载均衡，我们可以使用**均值不等式**对它进行一个**粗浅的理解(我个人的理解)**:

我们可以简单认为，每个专家分配的token比例和所有token选择它的平均概率相同，即$P_i = f_i$，那么损失就简化成:

$$
\text{Auxiliary Loss} = \alpha \cdot N \sum_{i=1}^N P_i^2, \quad \sum_{i=1}^N P_i = 1
$$

根据**QM-AM不等式**，我们有:

$$
\sqrt{\frac{\sum_{i=1}^n a_i^2}{n}} \geq \frac{\sum_{i=1}^n a_i}{n}
$$
(当且仅当所有$a_i$相同时取等)

带入，有:

$$
\sqrt{\frac{\sum_{i=1}^N P_i^2}{N}} \geq \frac{\sum_{i=1}^N P_i}{N} = \frac{1}{N}
$$

两边平方再乘上$\alpha \cdot N^2$，有:

$$
\begin{align*}
\alpha \cdot N^2 \cdot \left(\sqrt{\frac{\sum_{i=1}^N P_i^2}{N}}\right)^2

&= \alpha \cdot N \sum_{i=1}^N P_i^2

\\&\geq \alpha \cdot N^2 \cdot \left(\frac{1}{N}\right)^2

\\&= \alpha
\end{align*}
$$

当且仅当所有$P_i$相同时，即所有专家被选择的平均概率相同时，损失最低


### 参数估计

通过前面的分析我们可以知道，MoE的总参数量(稀疏参数)会更大，但是推理时只会激活其中一部分参数(激活参数)，也就是说，加载模型时我们仍需要全量加载，但是推理是需要激活部分参数，因此推理运行会更快，可以用**Mixtral 8×7B**为例计算一下它的激活参数和稀疏参数:

<div align="center">
    <img src="./imgs/para_num1.png" alt="para_num1" style="width: 500px; height: auto;">
</div>

我们逐层计算:

1. Embedding: 词表大小为32000，嵌入维度4096，那么总参数量就是$32000 \times 4096 = 131,072,000$

2. Attention: 32层decoder block，每个头维度为128，查询头有32个，KV头各有8个(**Grouped Query Attention**)，输出头也是32个，总参数量就是$32 \times (32+8+8+32) \times 128 \times 4096 = 1,342,177,280$

3. Router: 专家数量8个，也就是把4096维的输入映射为8维的输出，总参数量为$8 \times 4096 = 32,768$

4. Experts: 
    - 稀疏参数: 即总参数量，中间维度为14336，那么一个专家的参数量是$4096 \times 14336 \times 3(多一个门控) = 176,160,768$，32层，每层8个专家，那么总参数量为$176160768 \times 32 \times 8 = 45,097,156,608$

    - 激活参数: 也就是每个token在每层只会被2个专家处理，那么参数量就是$176160768 \times 32 \times 2 = 11,274,289,152$

5. LM Head(语言模型头): 与Embedding共享权重


也就是说，我们需要加载约45.1B的专家参数，但是实际计算时只会用到约11.3B的专家参数(Top2)

## 代码实现

Expert实现

就是一个MLP

In [91]:
import torch 
import torch.nn as nn
import torch.nn.functional as F


class Expert(nn.Module):
    def __init__(self, dim:int = 512, mlp_ratio:float = 4.0, dropout:float = 0.1):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(dim, int(mlp_ratio*dim)),
            nn.ReLU(),
            nn.Linear(int(mlp_ratio*dim), dim),
            nn.Dropout(dropout)
        )        


    def forward(self, x:torch.Tensor):
        return self.mlp(x)


带噪声的TopKRouter实现

这里说明一下，噪声的采样使用了**重参数化技巧**:

$$
\text{noise} = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$

这里我们$\mu$取0，$\sigma$由网络预测，具体预测方式是先由**线性层输出**每个expert的噪声强度$\beta$，再进行**softplus**保证标准差$\sigma$为正，其中softplus公式如下:

$$
\text{softplu}s(x) = \log(1 + e^x)
$$

那么最后的噪声就是:

$$
\beta = \text{Linear}(\text{input})

\\

\text{noise} = \text{softplus}(\beta) \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$


In [ ]:
class TopKRouter(nn.Module):
    def __init__(self, dim:int = 512, num_experts:int = 4, top_k:int = 2, 
                 auxiliary_loss_alpha:float = 0.01):
        super().__init__()

        self.top_k = top_k
        self.num_experts = num_experts
        self.linear = nn.Linear(dim, num_experts)
        # 添加噪声控制logits，输出的是噪声强度，用于生成sigma
        self.noise_linear = nn.Linear(dim, num_experts)
        self.auxiliary_loss_alpha = auxiliary_loss_alpha

    def forward(self, x:torch.Tensor, if_train=False, vis_loss=False):
        # [bs, seq_len, dim] -> [bs, seq_len, num_experts]
        logits = self.linear(x)
        noise_logits = self.noise_linear(x)
        noise = torch.randn_like(logits)*F.softplus(noise_logits)  # softplus保证标准差为正
        logits = logits + noise
        # -> [bs, seq_len, top_k], [bs, seq_len, top_k]
        top_k_logits, indices = logits.topk(self.top_k, dim=-1)
        # [bs, seq_len, num_experts]
        zeros = torch.full_like(logits, float('-inf'))
        # 按照列索引，把每个token对应的topk logits填入对应位置
        output_logits = zeros.scatter(-1, indices, top_k_logits)
        output_logits_softmax = F.softmax(output_logits, dim=-1)

        if if_train:
            if not vis_loss:
                auxiliary_loss = self._compute_auxiliary_loss(logits, indices)
            else:
                auxiliary_loss = self._compute_auxiliary_loss_visualize(logits, indices)
            return output_logits_softmax, indices, auxiliary_loss
            
        # -> [bs, seq_len, num_experts], [bs, seq_len, top_k]
        return output_logits_softmax, indices
    
    def _compute_auxiliary_loss(self, logits, indices):
        # [bs, seq_len, num_experts]
        router_probs = F.softmax(logits, dim=-1)
        # 转为one-hot编码，变为[bs, seq_len, top_k, num_experts]
        expert_mask = F.one_hot(indices, num_classes=self.num_experts)
        # [bs, seq_len, num_experts]，对每个token的top-k专家平均
        expert_mask = expert_mask.float().mean(dim=2)
        # [num_experts]，对所有token再平均
        frequency = expert_mask.mean(dim=[0, 1])
        
        # [num_experts]
        mean_router_probs = router_probs.mean(dim=[0, 1])

        auxiliary_loss = (frequency * mean_router_probs).sum() * self.num_experts

        return auxiliary_loss*self.auxiliary_loss_alpha
    
    def forward_visualize(self, x:torch.Tensor):
        # [bs, seq_len, dim] -> [bs, seq_len, num_experts]
        logits = self.linear(x)
        noise_logits = self.noise_linear(x)
        noise = torch.randn_like(logits)*F.softplus(noise_logits)
        logits = logits + noise
        print(f'init logits: \n{logits}')

        # -> [bs, seq_len, top_k], [bs, seq_len, top_k]
        top_k_logits, indices = logits.topk(self.top_k, dim=-1)
        print(f'top_k_logits: \n{top_k_logits}')
        print(f'indices: \n{indices}')

        # [bs, seq_len, num_experts]
        zeros = torch.full_like(logits, float('-inf'))
        # 按照列索引，把每个token对应的topk logits填入对应位置
        output_logits = zeros.scatter(-1, indices, top_k_logits)
        output_logits_softmax = F.softmax(output_logits, dim=-1)
        print(f'output_logits_softmax: \n{output_logits_softmax}')

        # -> [bs, seq_len, num_experts], [bs, seq_len, top_k]
        return output_logits_softmax, indices
    
    def _compute_auxiliary_loss_visualize(self, logits, indices):
        print(f'input logits: \n{logits}')
        print(f'input indices: \n{indices}')

        # [bs, seq_len, num_experts]
        router_probs = F.softmax(logits, dim=-1)
        # 转为one-hot编码，变为[bs, seq_len, top_k, num_experts]
        expert_mask = F.one_hot(indices, num_classes=self.num_experts)
        print(f'one-hot expert_mask: \n{expert_mask}')
        # [bs, seq_len, num_experts]，对每个token的top-k专家平均
        expert_mask = expert_mask.float().mean(dim=2)
        print(f'dim 2 mean, expert_mask: \n{expert_mask}')
        # [num_experts]，对所有token再平均
        frequency = expert_mask.mean(dim=[0, 1])
        print(f'frequency: \n{frequency}')
        
        # [num_experts]
        mean_router_probs = router_probs.mean(dim=[0, 1])
        print(f' mean_router_probs: \n{ mean_router_probs}')

        auxiliary_loss = (frequency * mean_router_probs).sum() * self.num_experts
        print(f'output loss: \n{auxiliary_loss*self.auxiliary_loss_alpha}')

        return auxiliary_loss*self.auxiliary_loss_alpha


In [93]:
top_k_router_example = TopKRouter(dim=5, num_experts=4, top_k=2)
top_k_router_example.eval()
x = torch.randn(2, 2, 5)
with torch.no_grad():
    _ = top_k_router_example.forward_visualize(x)


init logits: 
tensor([[[-0.1655,  0.6452,  0.1045, -1.2152],
         [-1.2782,  0.0899,  0.2919,  0.2218]],

        [[ 0.3590, -1.2335,  0.6691,  2.0440],
         [ 1.0002, -0.3520, -0.8168,  0.0171]]])
top_k_logits: 
tensor([[[0.6452, 0.1045],
         [0.2919, 0.2218]],

        [[2.0440, 0.6691],
         [1.0002, 0.0171]]])
indices: 
tensor([[[1, 2],
         [2, 3]],

        [[3, 2],
         [0, 3]]])
output_logits_softmax: 
tensor([[[0.0000, 0.6320, 0.3680, 0.0000],
         [0.0000, 0.0000, 0.5175, 0.4825]],

        [[0.0000, 0.0000, 0.2018, 0.7982],
         [0.7277, 0.0000, 0.0000, 0.2723]]])


In [104]:
top_k_router_example.train()
_ = top_k_router_example(x, if_train=True, vis_loss=True)


input logits: 
tensor([[[-0.2797, -0.5791, -0.2848,  0.3192],
         [-1.6662, -1.0118, -2.2147,  0.4187]],

        [[ 0.2664,  1.2857,  0.2081, -0.6579],
         [ 0.2226,  0.2803, -0.8482,  0.1043]]], grad_fn=<AddBackward0>)
input indices: 
tensor([[[3, 0],
         [3, 1]],

        [[1, 0],
         [1, 0]]])
ont-hot expert_mask: 
tensor([[[[0, 0, 0, 1],
          [1, 0, 0, 0]],

         [[0, 0, 0, 1],
          [0, 1, 0, 0]]],


        [[[0, 1, 0, 0],
          [1, 0, 0, 0]],

         [[0, 1, 0, 0],
          [1, 0, 0, 0]]]])
dim 2 mean, expert_mask: 
tensor([[[0.5000, 0.0000, 0.0000, 0.5000],
         [0.0000, 0.5000, 0.0000, 0.5000]],

        [[0.5000, 0.5000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000, 0.0000]]])
frequency: 
tensor([0.3750, 0.3750, 0.0000, 0.2500])
 mean_router_probs: 
tensor([0.2014, 0.2984, 0.1393, 0.3609], grad_fn=<MeanBackward1>)
output loss: 
0.011106050573289394


稀疏MoE模块

In [97]:
class SparseMoE(nn.Module):
    def __init__(self, dim:int = 512, num_experts:int = 4, top_k:int = 2):
        super().__init__()  

        self.router = TopKRouter(dim, num_experts, top_k)
        self.experts = nn.ModuleList([Expert(dim) for _ in range(num_experts)])
        self.top_k = top_k

    def forward(self, x:torch.Tensor):
        # -> [bs, seq_len, num_experts], [bs, seq_len, top_k]
        gating_output, indices = self.router(x)
        # [bs, seq_len, dim]
        final_output = torch.zeros_like(x)

        # 拼接所有batch的输入x和门控输出权重gating_output
        # -> [bs*seq_len, dim]
        flatten_x = x.view(-1, x.size(-1))
        # -> [bs*seq_len, num_experts]
        flatten_gating_output = gating_output.view(-1, gating_output.size(-1))

        for i, expert in enumerate(self.experts):
            # -> [bs, seq_len]，即每个token是否经过这个专家(True/False)
            expert_mask = (indices == i).any(dim=-1)
            # [bs*seq_len]，依旧是batch拼接
            flatten_expert_mask = expert_mask.view(-1)
            
            if flatten_expert_mask.any():  # 如果有token需要经过这个专家
                # -> [num_needed_token, dim]
                expert_input_x = flatten_x[flatten_expert_mask]
                expert_output_x = expert(expert_input_x)

                # -> [num_needed_token, 1](每个token对应当前专家的权重)
                gating_scores = flatten_gating_output[flatten_expert_mask, i].unsqueeze(1)
                # gating_scores广播为[num_needed_token, dim]计算
                weighted_output = expert_output_x * gating_scores  

                # final_output[expert_mask]选出来的形状是[num_needed_token, dim]，与weighted_output一致
                final_output[expert_mask] += weighted_output

        return final_output

    def forward_visualize(self, x:torch.Tensor):
        # -> [bs, seq_len, num_experts], [bs, seq_len, top_k]
        gating_output, indices = self.router(x)
        # [bs, seq_len, dim]
        final_output = torch.zeros_like(x)

        # 拼接所有batch的输入x和门控输出权重gating_output
        # -> [bs*seq_len, dim]
        flatten_x = x.view(-1, x.size(-1))
        # -> [bs*seq_len, num_experts]
        flatten_gating_output = gating_output.view(-1, gating_output.size(-1))
        print(f'flatten_x size: \n{flatten_x.size()}')
        print(f'flatten_gating_output size: \n{flatten_gating_output.size()}')

        for i, expert in enumerate(self.experts):
            print('\n', '='*114)
            print(f'Expert {i}:')
            # -> [bs, seq_len]，即每个token是否经过这个专家(True/False)
            expert_mask = (indices == i).any(dim=-1)
            # [bs*seq_len]，依旧是batch拼接
            flatten_expert_mask = expert_mask.view(-1)
            print(f'expert_mask: \n{expert_mask}')
            print(f'flatten_expert_mask: \n{flatten_expert_mask}')
            
            if flatten_expert_mask.any():  # 如果有token需要经过这个专家
                # -> [num_needed_token, dim]
                expert_input_x = flatten_x[flatten_expert_mask]
                expert_output_x = expert(expert_input_x)
                print(f'expert_input_x: \n{expert_input_x}')
                print(f'expert_output_x: \n{expert_output_x}')

                # -> [num_needed_token, 1](每个token对应当前专家的权重)
                gating_scores = flatten_gating_output[flatten_expert_mask, i].unsqueeze(1)
                print(f'gating_scores: \n{gating_scores}')
                # gating_scores广播为[num_needed_token, dim]计算
                weighted_output = expert_output_x * gating_scores
                print(f'weighted_output: \n{weighted_output}') 

                # final_output[expert_mask]选出来的形状是[num_needed_token, dim]，与weighted_output一致
                print(f'chosen final_output: \n{final_output[expert_mask]}')
                print(f'chosen final_output size: \n{final_output[expert_mask].size()}')
                final_output[expert_mask] += weighted_output
        
        print(f'\nfinal_output size: \n{final_output.size()}')
        return final_output


In [96]:
sparse_moe_example = SparseMoE(dim=5, num_experts=4, top_k=2)
sparse_moe_example.eval()
with torch.no_grad():
    _ = sparse_moe_example.forward_visualize(x)


flatten_x size: 
torch.Size([4, 5])
flatten_gating_output size: 
torch.Size([4, 4])

Expert 0:
expert_mask: 
tensor([[ True, False],
        [ True, False]])
flatten_expert_mask: 
tensor([ True, False,  True, False])
expert_input_x: 
tensor([[-1.0355, -0.2783, -0.0808,  0.6076, -0.9642],
        [ 1.2565,  0.9177, -0.6332,  0.1266,  0.2895]])
expert_output_x: 
tensor([[ 0.0666, -0.3412,  0.2730, -0.4226, -0.3754],
        [-0.0255, -0.1306, -0.0286, -0.1074,  0.0239]])
gating_scores: 
tensor([[0.8928],
        [0.4654]])
weighted_output: 
tensor([[ 0.0594, -0.3046,  0.2437, -0.3772, -0.3351],
        [-0.0119, -0.0608, -0.0133, -0.0500,  0.0111]])
chosen final_output: 
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])
chosen final_output size: 
torch.Size([2, 5])

Expert 1:
expert_mask: 
tensor([[False,  True],
        [ True,  True]])
flatten_expert_mask: 
tensor([False,  True,  True,  True])
expert_input_x: 
tensor([[-2.0492, -0.0824, -1.3502, -1.6879,  1.0676],
        [ 